# 리포트 49 — 검출기가 실제로 쓰는 커널 그대로 모호함수를 그렸다

> ### 한 일
> **기준신호 하나가 거리-도플러 평면에 만드는 응답을 검출기와 같은 커널로 계산하고, 검출기의 거리도플러 출력과 대조해 두 값의 최대 편차를 쟀다.**

### 결과
1. 모호함수와 검출기 거리도플러 출력은 최대 0.144 dB ⟨outputs/report03_illuminators.json : detector_af_max_err_db.value⟩ 안에서 같다 (6 ⟨outputs/report03_illuminators.json : detector_af_max_err_db.n_cases⟩개 경우, −45 dB 이상 셀).
2. 거리 주엽은 $c/B_{ref}$ 예측의 89% ⟨outputs/verify_ambiguity.json : waveforms.wifi_G1.dR_ratio⟩ ~ 94% ⟨outputs/verify_ambiguity.json : waveforms.nr_G1.dR_ratio⟩ 다(G1 세 파형).
3. 도플러 주엽은 여섯 경우 모두 $1/T_{CPI}$ 의 1.47 ⟨outputs/verify_ambiguity.json : waveforms.wifi_G1.dF_ratio⟩배 근처이고, 이 배수는 파형이 아니라 slow-time Hann 창이 정한다(`src/passive_process.py:142`).
4. 부엽과 ±PRF 레플리카는 표준마다 다르다 — 2D 부엽 최대가 LTE -5.3 ⟨outputs/verify_ambiguity.json : waveforms.lte_G1.psl_2d_db⟩ · 5G -18.0 dB ⟨outputs/verify_ambiguity.json : waveforms.nr_G1.psl_2d_db⟩ 이고, 레플리카는 WiFi -0.00 ⟨outputs/verify_ambiguity.json : waveforms.wifi_G1.doppler_replica_db⟩ · LTE -23.27 dB ⟨outputs/verify_ambiguity.json : waveforms.lte_G1.doppler_replica_db⟩ 다.

### 방법

| 무엇을 | 어떻게 얻었나 |
|---|---|
| 커널 | 검출기가 쓰는 것과 **같은 커널**로 계산한다 — `benchmark/verify_ambiguity.py:150`, 검출기는 `src/passive_process.py:133` |
| 대조 방식 | 표준 × 점유 경우마다 −45 dB 이상 셀의 최대 편차를 재고 그 최대값을 싣는다 |
| 슬로타임 창 | 프레임과 프레임 사이 축에 Hann 창을 씌운다 — 도플러 주엽의 배수를 정하는 것이 이 창이다(`src/passive_process.py:142`) |
| 이 표의 PRF | **검출기 프레임률**이다. 물리 주기 기준의 접힘은 따로 잰다 |

### 재현

```bash
PYTHONPATH=src:benchmark ~/.venvs/py312/bin/python benchmark/verify_ambiguity.py
PYTHONPATH=src ~/.venvs/py312/bin/python src/build_part08_illuminators.py
```

| | |
|---|---|
| 출력 | `outputs/verify_ambiguity.json`, `outputs/report03_illuminators.json` |
| 소요 | ② GPU 1장 수 분 · ④ CPU 20초 안쪽 |

### 앞 편에서

| 어디서 | 무엇을 알고 와야 하나 |
|---|---|
| [편 47 «바이스태틱 거리 분해능은 c/B»](47_range-convention.ipynb) | $\Delta R_b$ 와 잡음대역 규약 |

---

## 모호함수는 검출기의 눈이다

모호함수 $\chi(\tau, f_d)$ 는 기준신호 하나가 거리-도플러 평면에 만드는 응답이다. 표적이 점 하나여도 검출기 화면에는 이 모양이 찍힌다.

우리가 그리는 것은 **검출기가 쓰는 것과 같은 커널**이고, 검출기의 거리도플러 출력과 최대 0.144 dB ⟨outputs/report03_illuminators.json : detector_af_max_err_db.value⟩ (6 ⟨outputs/report03_illuminators.json : detector_af_max_err_db.n_cases⟩개 경우, −45 dB 이상 셀) 안에서 같다. 따로 계산한 그림이 아니라 **검출기 자신의 눈**이라는 뜻이다.

## 주엽 — 닫힌형과 대조

거리 주엽(응답에서 가장 높이 솟은 가운데 봉우리)은 $c/B_{ref}$ 예측의 89% ⟨outputs/verify_ambiguity.json : waveforms.wifi_G1.dR_ratio⟩ ~ 94% ⟨outputs/verify_ambiguity.json : waveforms.nr_G1.dR_ratio⟩ 다(G1 세 파형).

도플러 주엽은 여섯 경우 모두 $1/T_{CPI}$ 의 1.47 ⟨outputs/verify_ambiguity.json : waveforms.wifi_G1.dF_ratio⟩배 근처이고, 이 배수는 파형이 아니라 **slow-time Hann 창**(프레임과 프레임 사이 축에 씌워 가장자리를 깎는 창)이 정한다.

![report03_f6_af_mainlobe](../outputs/figures/report03_f6_af_mainlobe.png)

**그림 1.** 측정한 모호함수 주엽이 닫힌형 예측과 몇 % 안에서 맞는가?

## 부엽과 도플러 레플리카

주엽 밖으로 새는 에너지는 두 가지로 나타난다. **부엽**은 강한 표적이 평면 다른 곳의 약한 표적을 덮는 정도이고, **±PRF 레플리카**는 무모호 속도를 넘은 표적이 되접혀 들어오는 세기다.

| 기준신호 | 2D 부엽 최대 | ±PRF 레플리카 | 프레임 내 시간점유 |
|---|---|---|---|
| WiFi VHT-LTF | -14.3 dB ⟨outputs/verify_ambiguity.json : waveforms.wifi_G1.psl_2d_db⟩ | -0.00 dB ⟨outputs/verify_ambiguity.json : waveforms.wifi_G1.doppler_replica_db⟩ | 0.4% ⟨outputs/verify_ambiguity.json : waveforms.wifi_G1.ref_time_duty⟩ |
| LTE CRS | -5.3 dB ⟨outputs/verify_ambiguity.json : waveforms.lte_G1.psl_2d_db⟩ | -23.27 dB ⟨outputs/verify_ambiguity.json : waveforms.lte_G1.doppler_replica_db⟩ | 42.9% ⟨outputs/verify_ambiguity.json : waveforms.lte_G1.ref_time_duty⟩ |
| 5G SSB | -18.0 dB ⟨outputs/verify_ambiguity.json : waveforms.nr_G1.psl_2d_db⟩ | -1.05 dB ⟨outputs/verify_ambiguity.json : waveforms.nr_G1.doppler_replica_db⟩ | 28.6% ⟨outputs/verify_ambiguity.json : waveforms.nr_G1.ref_time_duty⟩ |

## 레플리카를 정하는 것은 점유율이 아니다

레플리카의 세기를 정하는 것은 **에너지가 프레임 안에 얼마나 퍼져 있는가**다. CRS 처럼 프레임 전체에 흩어지면 위상이 상쇄돼 레플리카가 죽고, LTF·SSB 처럼 앞쪽에 뭉치면 그대로 남는다.

이 표의 PRF 는 **검출기 프레임률**이다. 물리 주기 기준의 접힘은 [편 50 «5G SSB 는 걷는 드론에서 접힌다»](50_doppler-fold.ipynb) 가 따로 잰다 — 그 편이 같은 표를 물리 반복률로 다시 세운다.

## 다음 단계

| 다음에 할 일 | 그러면 결정되는 것 | 어디서 |
|---|---|---|
| `benchmark/run_min_cell.py:74` 의 `frame_len()` 을 물리 SSB 주기(50 Hz ⟨outputs/verify_ambiguity.json : waveforms.nr_G1.physical.prf_physical_hz⟩)로 확장한다 | 검출기 프레임률(2000 Hz ⟨outputs/verify_ambiguity.json : waveforms.nr_G1.physical.prf_model_hz⟩)과 40 ⟨outputs/verify_ambiguity.json : waveforms.nr_G1.physical.ratio⟩배 벌어진 이 편의 표가 한 규약 위에 선다 | `benchmark/verify_ambiguity.py:108` |
| 부엽 최대를 표적 두 개가 있는 장면에서 다시 잰다 | 강한 표적이 약한 표적을 덮는 거리가 수치로 확정된다 | `benchmark/verify_ambiguity.py` |